# 🎒 내 두뇌가 내 자료로 답하게 — 로컬 AI + RAG 실습

**AI CITY BUILDERS · 나만의 인공지능 6강 실습 2**

4강에서 LM 스튜디오의 두뇌를 채팅창 밖으로 꺼내 **API 로 불렀습니다.** 오늘은 그 두뇌에 **내 자료(책가방)** 를 달아, 두뇌가 모르던 것을 내 자료를 보고 답하게 만듭니다. 이것이 **RAG** 입니다.

| 오늘 할 일 | |
|---|---|
| ① 두뇌 켜기 | 코랩 안에 「내 컴퓨터의 두뇌」를 하나 띄웁니다 (올라마 · 젬마4) |
| ② 그냥 묻기 | 두뇌가 모르는 걸 물으면? → 모른다고 하거나 **지어냅니다** |
| ③ 책가방 싸기 | 내 자료(.txt)를 카드로 자릅니다 |
| ④ 책가방 매기 | 카드를 두뇌의 프롬프트에 붙입니다 → **내 자료로 답합니다** |
| ⑤ 가방이 무거우면 | 질문과 관련된 카드만 골라 넣습니다 |
| ⑥ 출처 달기 | 어느 카드를 보고 답했는지 밝히게 합니다 |
| ⑦ 나만의 자비스 | 채팅 화면으로 대화합니다 |
| ⑧ 내 컴퓨터로 | 주소 한 줄만 바꾸면 LM 스튜디오에서 그대로 |

**새 모델은 없습니다.** 임베딩 모델도 쓰지 않습니다. 그래서 젬마·큐원·올라마, **어떤 두뇌에든** 그대로 붙습니다.

👉 먼저 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 켜 주세요. 그다음 칸을 위에서부터 ▶ 하나씩.

## ① 두뇌 켜기 — 코랩 안에 「내 컴퓨터」 하나

코랩은 구글이 잠깐 빌려주는 컴퓨터입니다. 여기에 **올라마(Ollama)** 를 깔고 **젬마4** 두뇌를 받습니다. 3강에서 LM 스튜디오로 받은 것과 같은 두뇌입니다.

처음 한 번은 두뇌를 받느라 **3~5분** 걸립니다.

In [ ]:
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time, requests
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('✅ 올라마 켜짐 (주소: http://127.0.0.1:11434)')

In [ ]:
두뇌 = "gemma4:e2b"     # 3강의 젬마4 · 다른 두뇌도 됩니다 (예: "qwen2.5:3b", "gemma3:4b")
!ollama pull {두뇌} 2>&1 | tail -1
print('✅ 두뇌 받기 끝:', 두뇌)

## 두뇌에 묻는 함수 — 4강과 똑같습니다

LM 스튜디오든 올라마든, 두뇌는 **`/v1/chat/completions`** 라는 같은 주소로 부릅니다. 그래서 이 함수는 **내 컴퓨터에서도 그대로** 씁니다.

- `system` : 두뇌의 성격과 **배운 것(책가방)** 이 들어가는 자리
- `user` : 내 질문

In [ ]:
import requests, re

서버 = "http://127.0.0.1:11434"     # 내 컴퓨터 LM 스튜디오면 "http://127.0.0.1:1234"
성격 = "너는 다정한 로컬 두뇌야. 한국어로 짧게 답해."

def 두뇌에묻기(질문, 시스템=성격):
    r = requests.post(서버 + "/v1/chat/completions", json={
        "model": 두뇌, "temperature": 0,
        "messages": [{"role": "system", "content": 시스템},
                     {"role": "user", "content": 질문}]}, timeout=300).json()
    if "choices" not in r:
        raise SystemExit(f"두뇌가 답하지 않습니다 → {r}")
    글 = r["choices"][0]["message"]["content"]
    for 끝 in ("<channel|>", "</think>"):        # 생각을 소리 내는 두뇌는 생각 부분을 걷어낸다
        if 끝 in 글:
            글 = 글.split(끝)[-1]
    return 글.strip()

print(두뇌에묻기("안녕! 너는 누구야? 한 문장으로."))

## ② 그냥 묻기 — 두뇌는 내 이야기를 모릅니다

두뇌는 인터넷의 글을 잔뜩 읽고 만들어졌습니다. 하지만 **내 강아지 이름, 우리 가게 영업시간, 내 회사 규칙** 은 읽은 적이 없습니다.

모르는 걸 물으면 어떻게 할까요? 잘 보세요. **「모른다」고 하면 착한 두뇌, 그럴듯하게 지어내면 할루시네이션** 입니다.

In [ ]:
질문들 = [
    "우리 집 강아지 이름이 뭐야?",
    "우리 가게는 월요일에 문 열어?",
    "제이 멘토 강의 이북 서재 주소 알려줘.",
]
for q in 질문들:
    print("❓", q)
    print("🤖", 두뇌에묻기(q))
    print()

## ③ 책가방 싸기 — 내 자료를 카드로

책가방은 그냥 **폴더** 입니다. 안에 **.txt** 파일을 넣습니다. 메모, 가게 안내문, 회사 규칙, 일기… 무엇이든 됩니다.

아래 칸은 예시 파일 세 개를 만듭니다. **내 파일을 쓰고 싶으면** 그다음 칸에서 올리세요.

In [ ]:
import os
os.makedirs("책가방", exist_ok=True)

open("책가방/나의메모.txt", "w", encoding="utf-8").write(
"""강아지 이름은 소미이고, 고양이 이름은 나비입니다. 소미는 산책을 아주 좋아합니다.
자전거 자물쇠 번호는 4471 입니다.
엄마 생신은 11월 3일입니다. 선물로 머플러를 사 드리기로 했습니다.""")

open("책가방/우리가게.txt", "w", encoding="utf-8").write(
"""우리 가게 이름은 「소미네 커피」입니다.
영업시간은 화요일부터 일요일까지 오전 9시부터 오후 8시까지입니다. 월요일은 쉽니다.
가장 잘 팔리는 메뉴는 흑임자 라테이고 가격은 5,500원입니다.
주차는 가게 뒤편에 2대까지 됩니다.""")

open("책가방/강의안내.txt", "w", encoding="utf-8").write(
"""AI CITY BUILDERS 「나만의 인공지능」 과정 안내.
이북 서재 주소는 aicitybuilders.com/ebooks 입니다.
6강은 두뇌에 책가방을 다는 RAG 실습입니다. 새 모델 없이 3강 두뇌 하나로 합니다.
질문은 강의 페이지 아래 댓글이나 멤버십 라이브에서 받습니다.""")

for f in sorted(os.listdir("책가방")):
    print("📄", f, len(open("책가방/" + f, encoding="utf-8").read()), "자")

In [ ]:
# (선택) 내 .txt 파일을 책가방에 넣기 — 창이 안 뜨면 왼쪽 📁 폴더 아이콘 → 책가방 폴더에 끌어다 놓아도 됩니다
try:
    from google.colab import files
    올린것 = files.upload()
    for 이름, 내용 in 올린것.items():
        open("책가방/" + 이름, "wb").write(내용)
        print("🎒 넣음:", 이름)
except Exception as e:
    print("건너뜀:", e)

### 카드로 자르기

긴 글을 통째로 두는 것보다, **작은 카드** 로 잘라 두면 나중에 필요한 카드만 골라 넣을 수 있습니다. 문장 단위로 끊어서 **300자까지 한 장** 입니다.

In [ ]:
import glob

def 책가방싸기(폴더="책가방", 한장=300):
    카드 = []
    for 파일 in sorted(glob.glob(폴더 + "/*.txt") + glob.glob(폴더 + "/*.md")):
        글 = open(파일, encoding="utf-8").read()
        지금 = ""
        for 문장 in [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", 글) if s.strip()]:
            if 지금 and len(지금) + len(문장) > 한장:
                카드.append(지금); 지금 = ""
            지금 = (지금 + " " + 문장).strip()
        if 지금:
            카드.append(지금)
    return 카드

카드 = 책가방싸기()
print(f"🎒 카드 {len(카드)}장 · {sum(map(len, 카드))}자\n")
for i, c in enumerate(카드, 1):
    print(f"[카드 {i}] {c}")

## ④ 책가방 매기 — 카드를 프롬프트에 붙인다

RAG 의 핵심은 이게 전부입니다. **카드를 두뇌의 `system` 자리에 붙여서** 질문과 함께 보냅니다.

두뇌를 다시 학습시키지 않았습니다. **읽을거리를 옆에 놓아 준 것** 뿐입니다.

In [ ]:
def 책가방프롬프트(넣을카드):
    return (성격 + "\n\n## 내가 배운 것 (관련 있을 때만 참고해서 답해라)\n"
            + "\n".join("- " + c for c in 넣을카드))

print("━━━━ 두뇌에게 실제로 가는 system 프롬프트 ━━━━")
print(책가방프롬프트(카드))

In [ ]:
for q in 질문들:
    print("❓", q)
    print("   🎒 벗고:", 두뇌에묻기(q))
    print("   🎒 매고:", 두뇌에묻기(q, 책가방프롬프트(카드)))
    print()

같은 두뇌, 같은 질문인데 **책가방을 매니 답이 달라졌습니다.** 이게 RAG(Retrieval-Augmented Generation, 찾아서 붙여 생성하기)입니다.

> 💡 책가방에 **없는** 걸 물으면? 한번 해 보세요: `두뇌에묻기("우리 가게 와이파이 비밀번호는?", 책가방프롬프트(카드))`
> 좋은 두뇌는 「자료에 없다」고 답합니다. 지어내면, 성격 문장에 **「자료에 없으면 모른다고 답해」** 를 넣어 보세요.

## ⑤ 가방이 무거우면 — 관련 카드만 고르기

카드가 몇 장이면 통째로 넣어도 됩니다. 그런데 **수백, 수천 장** 이 되면 두뇌가 한 번에 읽을 수 있는 양(컨텍스트)을 넘고, 느려집니다.

그래서 **질문과 단어가 겹치는 카드부터** 골라 넣습니다. 임베딩 모델 없이, 단어만 세서 고릅니다.

아래 칸은 가방을 일부러 무겁게 만듭니다 (가짜 카드 300장 + 진짜 카드).

In [ ]:
import random
random.seed(0)
주제 = ["날씨", "점심 메뉴", "운동 기록", "읽은 책", "회의 메모", "장보기 목록", "여행 계획", "영화 감상"]
가짜 = [f"{random.choice(주제)} 메모 {i}번. 오늘은 특별한 일이 없었고, 평소처럼 하루를 보냈다." for i in range(300)]
무거운가방 = 가짜 + 카드
random.shuffle(무거운가방)
print(f"🎒 무거운 가방: 카드 {len(무거운가방)}장 · {sum(map(len, 무거운가방)):,}자")

In [ ]:
def 추리기(질문, 카드들, 장수=5):
    단어들 = [w for w in re.sub(r"[?？!！.,]", " ", 질문).split() if len(w) > 1]
    점수표 = sorted(((sum(1 for w in 단어들 if w in c), c) for c in 카드들), key=lambda x: -x[0])
    return 단어들, [(s, c) for s, c in 점수표[:장수] if s > 0]

q = "우리 가게 주차 되나요?"
단어들, 고른것 = 추리기(q, 무거운가방)
print("❓", q)
print("🔎 질문 단어:", 단어들)
print("━━━━ 골라낸 카드 (겹친 단어 수) ━━━━")
for s, c in 고른것:
    print(f"  {s}개 겹침 │ {c[:70]}")
print()
print("🤖", 두뇌에묻기(q, 책가방프롬프트([c for s, c in 고른것])))

> 🧪 **실험**: 「주차 가능해?」처럼 **단어를 바꿔** 물어보세요. 「가게」라는 단어가 없으면 못 찾을 수도 있습니다.
> 이것이 단어 세기의 한계입니다. 그래서 6강 다음 흐름에서 **판단하는 두뇌(JEV 같은 선택 모델)** 가 「무엇을 찾아볼지」를 고르게 됩니다.

## ⑥ 출처 달기 — 어느 카드를 보고 답했나

회사에서 쓰려면 **근거** 가 중요합니다. 카드에 번호를 붙이고, 답 끝에 **어느 카드를 봤는지** 적게 합니다.

In [ ]:
def 출처있게묻기(질문, 카드들):
    목록 = "\n".join(f"[카드 {i}] {c}" for i, c in enumerate(카드들, 1))
    시스템 = (성격 + " 아래 카드만 보고 답해. 카드에 없으면 「자료에 없어요」라고 해."
             " 답 끝에 참고한 카드 번호를 (근거: 카드 N) 처럼 적어.\n\n" + 목록)
    return 두뇌에묻기(질문, 시스템)

for q in ["흑임자 라테 얼마야?", "엄마 생신 선물 뭐로 하기로 했지?", "우리 가게 와이파이 비밀번호는?"]:
    print("❓", q)
    print("🤖", 출처있게묻기(q, 카드))
    print()

## ⑦ 나만의 자비스 — 채팅 화면으로

4강에서 만든 자비스처럼 채팅 화면을 띄웁니다. **책가방 매기** 를 켜고 끄며 비교해 보세요. 아래에 뜨는 화면에서 바로 대화합니다.

In [ ]:
!pip -q install gradio > /dev/null 2>&1
import gradio as gr

def 자비스(질문, 기록, 책가방매기):
    if not 책가방매기:
        return 두뇌에묻기(질문)
    전체 = sum(map(len, 카드))
    넣을것 = 카드 if 전체 <= 8000 else [c for s, c in 추리기(질문, 카드, 30)[1]]
    return 두뇌에묻기(질문, 책가방프롬프트(넣을것))

gr.ChatInterface(
    자비스,
    additional_inputs=[gr.Checkbox(value=True, label="🎒 책가방 매기")],
    title="🎒 나만의 자비스 — 내 자료로 답하는 로컬 두뇌",
    examples=[["우리 집 강아지 이름은?", True], ["가게 월요일에 열어?", True], ["자전거 자물쇠 번호?", False]],
).launch(share=False, debug=False)

## ⑧ 내 컴퓨터로 가져가기

이 코랩의 코드는 **내 컴퓨터에서 그대로** 돕니다. 바꿀 것은 두 줄뿐입니다.

| | 코랩 (오늘) | 내 컴퓨터 · LM 스튜디오 | 내 컴퓨터 · 올라마 |
|---|---|---|---|
| `서버` | `http://127.0.0.1:11434` | `http://127.0.0.1:1234` | `http://127.0.0.1:11434` |
| `두뇌` | `gemma4:e2b` | LM 스튜디오 목록의 이름 (예: `google/gemma-4-e2b`) | `gemma4:e2b` |

- LM 스튜디오: 왼쪽 **Developer** → **Start Server** 를 켜 두세요 (4강에서 한 것).
- 완성본 **rag.py** 와 설치 방법은 **6강 교재** 에 있습니다 → aicitybuilders.com/la6
- 4강에서 만든 자비스 앱에 붙이려면: 안티그래비티에 이 코랩의 `책가방싸기`·`책가방프롬프트`·`두뇌에묻기` 세 함수를 보여 주고 「내 자비스가 질문할 때마다 책가방 카드를 system 에 붙여서 묻게 고쳐 줘」라고 하면 됩니다.

---
### ✅ 완료 기준

- [ ] 책가방 벗고/매고 답이 달라지는 걸 봤다
- [ ] 내 .txt 파일을 하나 넣고, 그 내용을 물어서 맞게 답했다
- [ ] 무거운 가방에서 관련 카드를 골라내는 걸 봤다
- [ ] 답 끝에 근거 카드 번호가 붙었다
- [ ] (선택) 내 컴퓨터 LM 스튜디오에서 주소만 바꿔 돌렸다

### 🧪 더 해 보기
1. 성격 문장을 「너는 소미네 커피 직원이야. 손님께 공손하게 답해」로 바꿔 보세요. → 가게 챗봇
2. 두뇌를 `qwen2.5:3b` 로 바꿔 같은 질문을 해 보세요. → **책가방은 어떤 두뇌에든 붙습니다**
3. `한장=300` 을 100, 1000 으로 바꿔 보세요. 카드 크기에 따라 무엇이 달라지나요?